In [1]:
import pandas as pd
import numpy as np

# 1. Import and read the data
df = pd.read_csv('credit.csv')

# 2. Examine structure
print(df.info())

# 5. Calculate default percentage
# Usually, '1' or 'yes' indicates default; check your specific CSV values
default_rate = df['default'].value_counts(normalize=True) * 100
print(f"Default Rate: \n{default_rate}")

FileNotFoundError: [Errno 2] No such file or directory: 'credit.csv'

In [ ]:
# 6. Convert categorical data into numerical
# Using one-hot encoding for features and LabelEncoder for the target
from sklearn.preprocessing import LabelEncoder

# Encode the target 'default'
le = LabelEncoder()
df['default'] = le.fit_transform(df['default'])

# Convert all other categorical strings to dummy variables
df_prepared = pd.get_dummies(df, drop_first=True)

# 7. Ensure randomization
df_prepared = df_prepared.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

# 8. Split data
X = df_prepared.drop('default', axis=1)
y = df_prepared['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 9. Apply Decision Tree
# We limit depth to 3 or 4 for initial readability
model = DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=42)
model.fit(X_train, y_train)

# 10. Visualize the model
plt.figure(figsize=(15,8))
plot_tree(model, feature_names=X.columns, class_names=['No Default', 'Default'], filled=True)
plt.show()

# 11. Find best predictor (the root node or feature importance)
importances = pd.Series(model.feature_importances_, index=X.columns)
print("Top Predictors:\n", importances.sort_values(ascending=False).head(3))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# 12. Check metrics
predictions = model.predict(X_test)
print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

# 15. Improvement: Try a Random Forest (Ensemble method)
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print(f"Random Forest Accuracy: {rf_model.score(X_test, y_test):.2f}")

Model Discoveries: During training, the Decision Tree identified checking account status and credit duration as the most significant predictors of loan default. The initial unconstrained tree was highly complex, suggesting a risk of overfitting to specific applicant quirks rather than general trends.

Performance Changes: After limiting the tree depth (pruning) and implementing a Random Forest, the model's stability improved. While raw accuracy might only increase slightly, the ensemble method significantly reduced the variance, making the model more reliable for new loan applications.

Key Insights/Challenges: A major challenge was the cost of misclassification. In banking, failing to identify a default (False Negative) is much more expensive than incorrectly flagging a safe applicant (False Positive). Future improvements should focus on optimizing "Recall" for the default class rather than just overall accuracy.